# 01 — Slide inspection and physical magnification

RocqiPath treats `20x`, `40x` and `80x` as physical objective
magnifications. A pyramid level is not a magnification: level 1 is 10x on
one scanner and 5x on another. This notebook opens a slide, shows how the
scan magnification is resolved, and reads a region at an exact target
magnification.

A synthetic TIFF is created first so every cell runs without private data.
Set `USE_SYNTHETIC_DEMO = False` and point `SLIDE_PATH` at an SVS/NDPI/TIFF
to inspect your own slide.

In [ ]:
from pathlib import Path


def find_project_root(start: Path | None = None) -> Path:
    """Find the repository whether Jupyter starts at its root or in how_to_use/."""
    here = (start or Path.cwd()).resolve()
    for candidate in (here, *here.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src" / "rocqipath").is_dir():
            return candidate
    return here


PROJECT_ROOT = find_project_root()
DATA_ROOT = PROJECT_ROOT / "data"          # your slides (kept out of git)
RESULTS_ROOT = PROJECT_ROOT / "results"    # outputs (kept out of git)
DEMO_ROOT = PROJECT_ROOT / "notebook_demo_outputs"  # synthetic examples

import rocqipath as rp

print(f"RocqiPath {rp.__version__}")
print(f"Data    : {DATA_ROOT}")
print(f"Results : {RESULTS_ROOT}")

In [ ]:
USE_SYNTHETIC_DEMO = True
SLIDE_PATH = DATA_ROOT / "wsi" / "example_he.svs"
TARGET_MAGNIFICATION = 20.0

# Only needed when the file has no objective metadata (plain TIFFs, PNGs).
SOURCE_MAGNIFICATION = 40.0

In [ ]:
import numpy as np
from PIL import Image, ImageDraw

if USE_SYNTHETIC_DEMO:
    rng = np.random.default_rng(42)
    canvas = Image.new("RGB", (2048, 1536), (246, 246, 246))
    draw = ImageDraw.Draw(canvas)
    draw.rounded_rectangle((180, 160, 1868, 1376), radius=180, fill=(210, 148, 178))
    for _ in range(500):
        x, y, r = int(rng.integers(220, 1828)), int(rng.integers(200, 1336)), int(rng.integers(4, 13))
        draw.ellipse((x - r, y - r, x + r, y + r), fill=(95, 60, 130))
    SLIDE_PATH = DEMO_ROOT / "slide_inspection" / "synthetic_he.tif"
    SLIDE_PATH.parent.mkdir(parents=True, exist_ok=True)
    canvas.save(SLIDE_PATH)
print(SLIDE_PATH)

## Inspect metadata and resolve the read plan

The scan magnification comes from, in order: an explicit
`source_magnification`, scanner metadata (`openslide.objective-power`,
`aperio.AppMag`, …), or a RocqiPath manifest beside the file (aligned slides
have one). The same information is available from the command line with
`rocqipath info SLIDE`.

In [ ]:
with rp.open_slide(SLIDE_PATH) as slide:
    print(f"Level-0 dimensions : {slide.dimensions}")
    print(f"Level downsamples  : {slide.level_downsamples}")
    print(f"Objective metadata : {slide.properties.get('openslide.objective-power', 'none')}")
    plan = slide.configure_magnification(TARGET_MAGNIFICATION, SOURCE_MAGNIFICATION)

print(f"Scan magnification  : {plan.base_magnification:g}x")
print(f"Target magnification: {plan.target_magnification:g}x")
print(f"Stored level read   : {plan.level} ({plan.native_magnification:g}x)")
print(f"Final resize factor : {plan.resize_factor:.3f}")

## Read a region in target coordinates

After `target_magnification` is set, locations and sizes are in pixels *at
that magnification*; RocqiPath maps them to the stored pyramid and resizes
once.

In [ ]:
import matplotlib.pyplot as plt

PATCH = 512
with rp.open_slide(SLIDE_PATH, target_magnification=TARGET_MAGNIFICATION,
                   source_magnification=SOURCE_MAGNIFICATION) as slide:
    width, height = slide.target_dimensions
    location = ((width - PATCH) // 2, (height - PATCH) // 2)
    region = slide.read_at_magnification(location, (PATCH, PATCH)).convert("RGB")

print(f"Target-grid size {width} x {height}; read {region.size} at {location}")
plt.imshow(region)
plt.title(f"{PATCH} px at {TARGET_MAGNIFICATION:g}x")
plt.axis("off")
plt.show()

In [ ]:
from rocqipath.io.magnification import build_magnification_plan

for base, target, downsamples in [(80.0, 20.0, (1, 2, 4, 8)), (40.0, 20.0, (1, 4, 16)), (20.0, 20.0, (1, 2, 4))]:
    p = build_magnification_plan(base, target, downsamples)
    print(f"{base:>4g}x scan -> {target:g}x: read level {p.level} ({p.native_magnification:g}x), resize {p.resize_factor:.3f}")

## Common problems

- **"objective magnification could not be resolved"** — pass
  `source_magnification=` with the objective the file was scanned at.
- **Target above the scan magnification** — refused; upsampling cannot add detail.
- **OpenSlide cannot open the file** — install OpenSlide (or `openslide-bin`);
  ordinary TIFFs fall back to Pillow.

Continue with **02** for tissue and TMA extraction.